# CRF NER for Cellphone Product Titles

This notebook builds a **Conditional Random Field (CRF)** named-entity recognizer
for Brazilian Portuguese e-commerce product titles (cellphones/accessories),
using `sklearn-crfsuite`. It is one of three NER techniques being compared for
this assignment (the other two: spaCy and BERTimbau: are separate).

Tags: `TIPO`, `MARCA`, `MODELO`, `MEMORIA`, `RAM`, `COR`, `TELA`.

All the actual logic (tokenizer, BIO conversion, feature extraction) lives in
`scripts/train_crf.py`: this notebook imports and calls that module so there
is a single source of truth for the pipeline, and reproduces the same run
inline with explanations.

## Setup

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if (Path.cwd() / "notebooks").exists() is False and Path.cwd().name == "notebooks" else Path.cwd()
# Ensure ROOT points at the project root (the directory containing `scripts/`)
if not (ROOT / "scripts").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from scripts import train_crf as crf_mod

print("Project root:", ROOT)


Project root: /Users/joaorietra/Developer/mt-lab-session-ner-starter


## 1. Tokenization

We use a regex tokenizer (`crf_mod.tokenize`) that:
- Keeps decimal numbers like `6,5` or `6.5` as one token (important for `TELA`).
- Keeps alphanumeric runs together, e.g. `128GB`, `A10s`, `13MP`.
- Splits punctuation (`-`, `/`, `"`, `,`, `+`, parentheses, etc.) into their own
  tokens.

This keeps unit-bearing tokens (`128GB`, `6,5`) intact for feature extraction
while not merging unrelated words across punctuation.

In [2]:
example_text = "Smartphone Samsung Galaxy A10s 32GB Android 9.0 Tela 6.2\u201d Octa-Core 4G C\u00e2mera 13MP+2MP - Preto"
tokens = crf_mod.tokenize(example_text)
tokens


[('Smartphone', 0, 10),
 ('Samsung', 11, 18),
 ('Galaxy', 19, 25),
 ('A10', 26, 29),
 ('s', 29, 30),
 ('32GB', 31, 35),
 ('Android', 36, 43),
 ('9.0', 44, 47),
 ('Tela', 48, 52),
 ('6.2', 53, 56),
 ('”', 56, 57),
 ('Octa', 58, 62),
 ('-', 62, 63),
 ('Core', 63, 67),
 ('4G', 68, 70),
 ('Câmera', 71, 77),
 ('13MP', 78, 82),
 ('+', 82, 83),
 ('2MP', 83, 86),
 ('-', 87, 88),
 ('Preto', 89, 94)]

## 2. Char-offset spans -> BIO tags

`crf_mod.spans_to_bio` walks each annotated `[start, end, TAG]` span and marks
every token **fully contained** inside it as `B-TAG`/`I-TAG`. A token that only
*partially* overlaps a span (misaligned annotation) is never silently
mislabeled: the example is logged to stderr and skipped entirely by
`crf_mod.load_dataset`.

In [3]:
tags = crf_mod.spans_to_bio(
    example_text,
    [[0, 10, "TIPO"], [11, 18, "MARCA"], [19, 30, "MODELO"], [31, 35, "MEMORIA"], [48, 57, "TELA"], [89, 94, "COR"]],
    tokens,
)
list(zip([t[0] for t in tokens], tags))


[('Smartphone', 'B-TIPO'),
 ('Samsung', 'B-MARCA'),
 ('Galaxy', 'B-MODELO'),
 ('A10', 'I-MODELO'),
 ('s', 'I-MODELO'),
 ('32GB', 'B-MEMORIA'),
 ('Android', 'O'),
 ('9.0', 'O'),
 ('Tela', 'B-TELA'),
 ('6.2', 'I-TELA'),
 ('”', 'I-TELA'),
 ('Octa', 'O'),
 ('-', 'O'),
 ('Core', 'O'),
 ('4G', 'O'),
 ('Câmera', 'O'),
 ('13MP', 'O'),
 ('+', 'O'),
 ('2MP', 'O'),
 ('-', 'O'),
 ('Preto', 'B-COR')]

## 3. Load train/test sets

In [4]:
train_sents, train_tags, train_texts = crf_mod.load_dataset(crf_mod.TRAIN_PATH)
test_sents, test_tags, test_texts = crf_mod.load_dataset(crf_mod.TEST_PATH)

len(train_sents), len(test_sents)


[train.jsonl] loaded 240 examples, skipped 0
[test.jsonl] loaded 60 examples, skipped 0


(240, 60)

## 4. Feature engineering

Per-token features (`crf_mod.word2features`), computed for the token itself and
a -1/+1 neighbor window, plus BOS/EOS flags:

- lowercased word, 2-3 char suffix/prefix
- word shape (case pattern, collapsed), e.g. `Galaxy` -> `Xx`, `128GB` -> `dX`
- flags: is uppercase, is title case, is fully digit, contains a digit
- unit-pattern flags: matches `gb/mb/tb/mp/pol/hz/mah/ram`, decimal pattern
  (`6,5`), numeric-plus-unit pattern (`128gb`)
- token length, pure-punctuation flag

In [5]:
crf_mod.word2features(tokens_as_words := [t[0] for t in tokens], 5)


{'bias': 1.0,
 'cur.word.lower': '32gb',
 'cur.word.shape': 'dX',
 'cur.word.suffix2': 'gb',
 'cur.word.suffix3': '2gb',
 'cur.word.prefix2': '32',
 'cur.word.isupper': True,
 'cur.word.istitle': False,
 'cur.word.isdigit': False,
 'cur.word.has_digit': True,
 'cur.word.is_unit': False,
 'cur.word.is_decimal': False,
 'cur.word.is_numeric_with_unit': True,
 'cur.word.len': 4,
 'cur.word.is_punct': False,
 '-1.word.lower': 's',
 '-1.word.shape': 'x',
 '-1.word.suffix2': 's',
 '-1.word.suffix3': 's',
 '-1.word.prefix2': 's',
 '-1.word.isupper': False,
 '-1.word.istitle': False,
 '-1.word.isdigit': False,
 '-1.word.has_digit': False,
 '-1.word.is_unit': False,
 '-1.word.is_decimal': False,
 '-1.word.is_numeric_with_unit': False,
 '-1.word.len': 1,
 '-1.word.is_punct': False,
 '+1.word.lower': 'android',
 '+1.word.shape': 'Xx',
 '+1.word.suffix2': 'id',
 '+1.word.suffix3': 'oid',
 '+1.word.prefix2': 'an',
 '+1.word.isupper': False,
 '+1.word.istitle': True,
 '+1.word.isdigit': False,
 '+1.

## 5. Build feature matrices

In [6]:
X_train = [crf_mod.sent2features(s) for s in train_sents]
y_train = train_tags
X_test = [crf_mod.sent2features(s) for s in test_sents]
y_test = test_tags

len(X_train), len(X_test)


(240, 60)

## 6. Train the CRF

`sklearn_crfsuite.CRF` with the `lbfgs` algorithm and light L1/L2 regularization
(`c1=0.1`, `c2=0.1`), 100 max iterations.

In [7]:
import sklearn_crfsuite

crf = sklearn_crfsuite.CRF(
    algorithm="lbfgs",
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True,
)
crf.fit(X_train, y_train)


,algorithm,'lbfgs'
,all_possible_transitions,True
,c1,0.1
,c2,0.1
,max_iterations,100
,min_freq,None
,all_possible_states,None
,num_memories,None
,epsilon,None
,period,None
,delta,None


## 7. Evaluate on the test set

Using `seqeval.metrics.classification_report` (entity-level, IOB2 scheme) :
this scores whole entity spans, not individual tokens.

In [8]:
from seqeval.metrics import classification_report as seq_classification_report
from seqeval.metrics import f1_score as seq_f1_score
from seqeval.scheme import IOB2

y_pred = crf.predict(X_test)

report_str = seq_classification_report(y_test, y_pred, scheme=IOB2, digits=4)
overall_f1 = seq_f1_score(y_test, y_pred, scheme=IOB2)

print(report_str)
print(f"Overall micro F1: {overall_f1:.4f}")


              precision    recall  f1-score   support

         COR     0.8810    0.8409    0.8605        44
       MARCA     0.9355    0.9667    0.9508        60
     MEMORIA     0.9706    1.0000    0.9851        33
      MODELO     0.9000    0.8060    0.8504        67
         RAM     1.0000    1.0000    1.0000        14
        TELA     0.8571    0.8571    0.8571        21
        TIPO     0.9388    0.8846    0.9109        52

   micro avg     0.9220    0.8935    0.9075       291
   macro avg     0.9261    0.9079    0.9164       291
weighted avg     0.9211    0.8935    0.9064       291

Overall micro F1: 0.9075


## 8. Save metrics

Reuses the exact same JSON artifact produced by `scripts/train_crf.py` at
`notebooks/results/crf_metrics.json`: running the script and running this
notebook converge on the same result file (single source of truth).

In [9]:
import json

report_dict = seq_classification_report(y_test, y_pred, scheme=IOB2, digits=4, output_dict=True)

def _json_default(o):
    if hasattr(o, "item"):
        return o.item()
    raise TypeError(f"Object of type {type(o).__name__} is not JSON serializable")

crf_mod.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with crf_mod.RESULTS_PATH.open("w", encoding="utf-8") as f:
    json.dump(
        {
            "overall_f1_micro": overall_f1,
            "report": report_dict,
            "n_train": len(train_sents),
            "n_test": len(test_sents),
        },
        f,
        ensure_ascii=False,
        indent=2,
        default=_json_default,
    )

print("Saved to", crf_mod.RESULTS_PATH)


Saved to /Users/joaorietra/Developer/mt-lab-session-ner-starter/notebooks/results/crf_metrics.json


## 9. Qualitative examples

Token-level gold vs. predicted BIO tags for a handful of test titles, for
narrative/presentation purposes.

In [10]:
n_show = min(8, len(test_sents))
for i in range(n_show):
    print(f"\n--- Example {i + 1} ---")
    print("Text:", test_texts[i])
    print(f"{'Token':<15}{'Gold':<12}{'Pred':<12}")
    for tok, gold, pred in zip(test_sents[i], y_test[i], y_pred[i]):
        marker = "" if gold == pred else "  <-- MISMATCH"
        print(f"{tok:<15}{gold:<12}{pred:<12}{marker}")



--- Example 1 ---
Text: Smartphone Nokia C20 32GB 4G com 90 dias de internet grátis* - NK081
Token          Gold        Pred        
Smartphone     B-TIPO      B-TIPO      
Nokia          B-MARCA     B-MARCA     
C20            B-MODELO    B-MODELO    
32GB           B-MEMORIA   B-MEMORIA   
4G             O           O           
com            O           O           
90             O           O           
dias           O           O           
de             O           O           
internet       O           O           
grátis         O           O           
*              O           O           
-              O           O           
NK081          O           O           

--- Example 2 ---
Text: KIT MOTORISTA 01 - WI392
Token          Gold        Pred        
KIT            B-TIPO      B-TIPO      
MOTORISTA      B-MODELO    O             <-- MISMATCH
01             I-MODELO    O             <-- MISMATCH
-              O           O           
WI392          O           O

## Summary

The CRF model reaches an overall (micro) F1 of **~0.91** on the held-out test
set, with strong performance on structurally regular tags (`RAM`, `MEMORIA`,
`MARCA`) and somewhat lower recall on `MODELO` and `COR`, which have more
lexical variety and occasional multi-token ambiguity (e.g. accessory
descriptions like "Base Carregadora" being confused with `TIPO`). See
`notebooks/results/crf_metrics.json` for the full per-tag breakdown.